# Fase 3: Data Preparation

En este notebook realizaremos una exploración simple del CSV bruto ubicado en `bruto/` y aplicaremos pasos iniciales de limpieza. Guardaremos un CSV intermedio en `intermedio/` con el resultado de la limpieza segura para que la fase de preparación pueda continuar y producir el CSV final en `procesado/`.

### Importaciones

In [45]:
# Imports y configuración inicial
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visual helpers (si se ejecuta en notebook)
from IPython.display import display

### Carga de dataset bruto

In [46]:
# Lectura del CSV bruto (desde la carpeta `bruto` dentro de este notebook)
base = Path('bruto')
csv_name = 'csv_ferredash_raw.csv'
csv_path = base / csv_name

if not csv_path.exists():
    # Buscador alternativo sencillo: buscar en el repo (solo si no existe la ruta esperada)
    print(f'No se encontró {csv_path}. Asegúrate que el CSV esté en la carpeta `bruto/`.')
else:
    print('Leyendo:', csv_path)
    df = pd.read_csv(csv_path, parse_dates=['fecha_venta'], low_memory=False)
    print('Leído. Shape:', df.shape)


Leyendo: bruto\csv_ferredash_raw.csv
Leído. Shape: (180515, 23)
Leído. Shape: (180515, 23)


## Exploración simple

Mostramos forma, primeras filas, tipos y un resumen de valores nulos para decidir la estrategia de limpieza e imputación.

In [47]:
# Exploración: shape, head, dtypes y nulos
if 'df' in globals():
    print('Shape:', df.shape)
    display(df.head())
    display(df.dtypes.to_frame('dtype'))
    missing = df.isnull().sum().rename('n_missing').to_frame()
    missing['pct_missing'] = 100 * missing['n_missing'] / len(df)
    display(missing.sort_values('pct_missing', ascending=False))
    # Estadísticas numéricas compactas
    display(df.select_dtypes(include=[np.number]).describe().T)
else:
    print('df no definido. Verifica la lectura del CSV.')

Shape: (180515, 23)


,ven_id,fecha_venta,anio,mes,dia_semana,prod_id,prod_nom,prod_marca,categoria,precio_venta,...,ingreso_neto,suc_id,sucursal,usu_id,vendedor,cli_id,cliente,stock_actual,valor_stock_pendiente,utilidad_acumulada
0,7,2025-12-09,2025,12,Tuesday,2323,CLAVO TECHO 1 3/4,NO POSEE,CLAVOS,4800.0,...,20500.0,1,Casa Matriz,2,Vendedor1,1,Cliente Estandar,17.0,68571.0,68571.0
1,16,2025-12-09,2025,12,Tuesday,5854,PUNTAS 1 1/2 ;500 GRS BOLSA,NO POSEE,CLAVOS,2900.0,...,2900.0,1,Casa Matriz,2,Vendedor1,1,Cliente Estandar,8.0,19495.0,19495.0
2,33,2025-12-09,2025,12,Tuesday,286,ALAMBRE 2.5MM ROJO MT,NO POSEE,ELECTRICO,450.0,...,2350.0,1,Casa Matriz,2,Vendedor1,1,Cliente Estandar,0.0,0.0,0.0
3,9,2025-12-09,2025,12,Tuesday,5616,PISTOLA DE CALOR 2000W,NO POSEE,HERRAMIENTAS ELECTRICAS,30000.0,...,36700.0,1,Casa Matriz,2,Vendedor1,1,Cliente Estandar,22.0,554621.0,554621.0
4,15,2025-12-09,2025,12,Tuesday,2661,COPIA DE LLAVE CASA,NO POSEE,HERRAMIENTAS,1800.0,...,3600.0,1,Casa Matriz,2,Vendedor1,1,Cliente Estandar,35.0,44117.0,44117.0


,dtype
ven_id,int64
fecha_venta,datetime64[ns]
anio,int64
mes,int64
dia_semana,object
prod_id,int64
prod_nom,object
prod_marca,object
categoria,object
precio_venta,float64


,n_missing,pct_missing
ven_id,0,0.0
fecha_venta,0,0.0
anio,0,0.0
mes,0,0.0
dia_semana,0,0.0
prod_id,0,0.0
prod_nom,0,0.0
prod_marca,0,0.0
categoria,0,0.0
precio_venta,0,0.0


,count,mean,std,min,25%,50%,75%,max
ven_id,180515.0,90258.000000,52110.336259,1.00,45129.500,90258.00,135386.500,180515.00
anio,180515.0,2024.173947,0.379065,2024.00,2024.000,2024.00,2024.000,2025.00
mes,180515.0,2.508717,3.268293,1.00,1.000,1.00,1.000,12.00
prod_id,180515.0,4256.239121,2506.317446,6.00,2301.000,4105.00,6724.000,8066.00
precio_venta,180515.0,3055.659363,4985.306680,0.00,750.000,2000.00,3900.000,325000.00
ven_cantidad,180515.0,7.411163,50.515703,0.00,1.000,2.00,4.000,5270.00
ven_descuento,180515.0,0.000000,0.000000,0.00,0.000,0.00,0.000,0.00
ven_precio_unitario,180515.0,4258.475903,27196.872634,29.10,1234.460,2700.00,4448.485,2396236.35
ingreso_neto,180515.0,12335.708168,100007.797703,9.01,2962.655,5383.24,9932.935,4304379.84
suc_id,180515.0,1.326095,0.468784,1.00,1.000,1.00,2.000,2.00


## Plan de limpieza inicial

Este bloque aplicará transformaciones seguras y reversibles:
- Normalizar strings (strip, upper/lower según convenga).
- Reemplazar marcadores de "sin dato" como 'NO POSEE', 'Sin Categoria', 'PRODUCTO NO IDENTIFICADO' por `NaN`.
- Convertir columnas numéricas mal tipadas con `pd.to_numeric(..., errors='coerce')`.
- Rellenar `ven_descuento` con 0 si aplica.
- Eliminar duplicados exactos.
- Guardar resultado intermedio en `intermedio/csv_intermedio_step1.csv`.

Nota: la imputación (mediana/media) se propone sólo si el porcentaje de nulos es pequeño; si es grande, conservaremos la columna para tratamiento manual o modelado que acepte nulos.

In [48]:
# Limpieza segura y guardado intermedio
if 'df' in globals():
    df_clean = df.copy()

    # Normalizar strings comunes
    str_cols = ['prod_nom','prod_marca','categoria','sucursal','vendedor','cliente']
    for c in str_cols:
        if c in df_clean.columns:
            df_clean[c] = df_clean[c].astype(str).str.strip()
            # Convertir placeholders a NaN
            df_clean[c] = df_clean[c].replace({'NO POSEE':np.nan,'Sin Categoria':np.nan,'PRODUCTO NO IDENTIFICADO':np.nan,'nan':np.nan})

    # Convertir columnas numéricas que pueden venir como strings
    num_candidates = ['precio_venta','ven_cantidad','ven_descuento','ven_precio_unitario','ingreso_neto','stock_actual','valor_stock_pendiente','utilidad_acumulada']
    for nc in num_candidates:
        if nc in df_clean.columns:
            df_clean[nc] = pd.to_numeric(df_clean[nc], errors='coerce')

    # Rellenar descuentos faltantes con 0 (si tiene sentido de negocio)
    if 'ven_descuento' in df_clean.columns:
        df_clean['ven_descuento'] = df_clean['ven_descuento'].fillna(0)

    # Eliminar duplicados exactos
    before = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    after = len(df_clean)
    print(f'Duplicados removidos: {before-after}')

    # Recalcular missing tras limpieza
    missing_after = df_clean.isnull().sum().rename('n_missing').to_frame()
    missing_after['pct_missing'] = 100 * missing_after['n_missing'] / len(df_clean)
    display(missing_after.sort_values('pct_missing', ascending=False))

    # Guardar intermedio
    inter_path = Path('intermedio')
    inter_path.mkdir(parents=True, exist_ok=True)
    out_file = inter_path / 'csv_intermedio_step1.csv'
    df_clean.to_csv(out_file, index=False)
    print('CSV intermedio guardado en:', out_file)

else:
    print('df no definido; no se ejecutó la limpieza')

Duplicados removidos: 0


,n_missing,pct_missing
prod_marca,180515,100.000000
categoria,21585,11.957455
prod_nom,17075,9.459048
anio,0,0.000000
mes,0,0.000000
fecha_venta,0,0.000000
ven_id,0,0.000000
prod_id,0,0.000000
dia_semana,0,0.000000
precio_venta,0,0.000000


CSV intermedio guardado en: intermedio\csv_intermedio_step1.csv


### Análisis de Limpieza de Datos — Resultados Paso 1

Tras ejecutar el primer proceso de limpieza, se obtuvieron los siguientes resultados y observaciones relevantes para entender el estado del dataset y los próximos pasos necesarios en el flujo CRISP-DM.

---

#### 1. Duplicados

- **Duplicados eliminados:** `0`
- **Interpretación:**  
  El dataset no contenía filas completamente duplicadas según todos los campos. Esto indica que:
  - El origen de los datos es consistente en su estructura.
  - Los problemas de calidad están más asociados a *faltantes* o *valores no estandarizados*, no a redundancia.

---

#### 2. Valores faltantes (`missing values`)

Se generó un ranking de columnas con datos faltantes:

| Columna | Faltantes | % |
|--------|-----------|--------------|
| **prod_marca** | 180.515 | 100% |
| categoria | 21.585 | 11.96% |
| prod_nom | 17.075 | 9.46% |
| *Todas las demás columnas* | 0 | 0% |

#### Interpretación analítica

##### **prod_marca — 100% faltante**
- Esta columna está completamente vacía.
- Posibles razones:
  - El origen de los datos nunca provee marca.
  - La marca está embebida dentro de otra columna (ej: `prod_nom`).
  - La marca no es relevante para el proceso de ventas o no se registra en el sistema original.

**Acción recomendada:**

- Evaluar **eliminar** la columna.
- O diseñar un proceso de **extracción de marca desde `prod_nom`** si el nombre contiene patrones como `"Martillo Stanley" → marca = Stanley"`.

---

##### **categoria — 11.96% faltante**
- La cantidad de faltantes aún es significativa.
- Podrían ser productos sin clasificación o casos donde venía `"NO POSEE"` o `"Sin Categoria"` y fueron convertidos a `NaN`.

**Acciones posibles:**
- Completar categorías mediante:
  - Reglas simples (ej: patrones en `prod_nom`).
  - Algoritmos NLP / clustering para sugerir categorías.
- O crear una categoría `"Sin Clasificar"` para análisis operacionales.

---

##### **prod_nom — 9.46% faltante**
- Un 9.5% de nombres de producto faltantes es crítico, ya que:
  - Afecta identificación del producto.
  - Impacta en modelos de demanda.
  - Impide agrupar o generar métricas correctas.

**Acciones recomendadas:**
- Validar si realmente están vacíos o venían como `"PRODUCTO NO IDENTIFICADO"`.
- Revisar relación con `prod_id`:
  - Si `prod_id` tiene registros repetidos, quizá el nombre se pueda imputar por correspondencia.

---

#### 3. Columnas numéricas convertidas

Las columnas numéricas:

- precio_venta  
- ven_cantidad  
- ven_descuento  
- ven_precio_unitario  
- ingreso_neto  
- stock_actual  
- valor_stock_pendiente  
- utilidad_acumulada  

fueron convertidas correctamente a números mediante `to_numeric(errors='coerce')`.

**Resultado:**
- No aparecen faltantes en estas columnas → la conversión fue exitosa.
- Los cálculos estadísticos y modelados posteriores serán precisos.

---

#### 4. Archivo Intermedio Guardado

Se generó un archivo intermedio en el proceso de limpieza.

### Conclusiones Generales

- La estructura del dataset es estable y sin duplicados.
- El principal problema detectado es **información faltante en atributos clave**, especialmente:
  - `prod_marca` → completamente vacío.
  - `categoria` y `prod_nom` → faltantes relevantes a nivel analítico.
- Antes de avanzar al modelado, será necesario abordar:
  - **Recuperación / imputación** de nombres y categorías.
  - **Decidir si se elimina `prod_marca`.**
- Todo está listo para avanzar hacia:
  - Exploración profunda (`EDA`)
  - Limpieza avanzada e imputación
  - Ingeniería de características
  - Preparación para modelado predictivo.


### Limpieza profunda

In [49]:
# - No altera la celda previa; escribe un CSV intermedio de variante para verificación
if 'df' in globals():
    df_clean2 = df.copy()

    # Normalizar strings comunes (sin prod_marca)
    str_cols = ['prod_nom','categoria','sucursal','vendedor','cliente']
    for c in str_cols:
        if c in df_clean2.columns:
            df_clean2[c] = df_clean2[c].astype(str).str.strip()
            df_clean2[c] = df_clean2[c].replace({'NO POSEE':np.nan,'Sin Categoria':np.nan,'PRODUCTO NO IDENTIFICADO':np.nan,'nan':np.nan})

    # Eliminar prod_marca si existe (no se altera el df original)
    if 'prod_marca' in df_clean2.columns:
        df_clean2 = df_clean2.drop(columns=['prod_marca'])
        print("Columna 'prod_marca' eliminada en df_clean2")

    # Convertir columnas numéricas que pueden venir como strings
    num_candidates = ['precio_venta','ven_cantidad','ven_descuento','ven_precio_unitario','ingreso_neto','stock_actual','valor_stock_pendiente','utilidad_acumulada']
    for nc in num_candidates:
        if nc in df_clean2.columns:
            df_clean2[nc] = pd.to_numeric(df_clean2[nc], errors='coerce')

    # Rellenar descuentos faltantes con 0 (si aplica)
    if 'ven_descuento' in df_clean2.columns:
        df_clean2['ven_descuento'] = df_clean2['ven_descuento'].fillna(0)

    # Imputación de 'categoria' usando vecino previo/siguiente según reglas:
    if 'categoria' in df_clean2.columns:
        prev = df_clean2['categoria'].ffill()
        nxt = df_clean2['categoria'].bfill()
        mask_null = df_clean2['categoria'].isna()
        same_mask = mask_null & prev.eq(nxt) & prev.notna()
        df_clean2.loc[same_mask, 'categoria'] = prev[same_mask]
        still_null = df_clean2['categoria'].isna()
        prefer_prev = still_null & prev.notna()
        df_clean2.loc[prefer_prev, 'categoria'] = prev[prefer_prev]
        prefer_next = df_clean2['categoria'].isna() & nxt.notna()
        df_clean2.loc[prefer_next, 'categoria'] = nxt[prefer_next]
        print("Imputación de 'categoria' aplicada en df_clean2 (prev/next).")

    # Imputación de 'prod_nom': por prod_id->moda, luego vecinos
    imputed_prod_nom = 0
    if 'prod_nom' in df_clean2.columns:
        if 'prod_id' in df_clean2.columns:
            mode_map = df_clean2.loc[df_clean2['prod_nom'].notna()].groupby('prod_id')['prod_nom'].agg(lambda x: x.mode().iat[0] if not x.mode().empty else np.nan)
            before_nulls = df_clean2['prod_nom'].isna().sum()
            df_clean2['prod_nom'] = df_clean2['prod_nom'].fillna(df_clean2['prod_id'].map(mode_map))
            after_map_nulls = df_clean2['prod_nom'].isna().sum()
            imputed_prod_nom += (before_nulls - after_map_nulls)
            # Rellenar por vecinos si queda nulo
            before_neighbors = df_clean2['prod_nom'].isna().sum()
            df_clean2['prod_nom'] = df_clean2['prod_nom'].ffill().bfill()
            after_neighbors = df_clean2['prod_nom'].isna().sum()
            imputed_prod_nom += (before_neighbors - after_neighbors)
        else:
            before_neighbors = df_clean2['prod_nom'].isna().sum()
            df_clean2['prod_nom'] = df_clean2['prod_nom'].ffill().bfill()
            imputed_prod_nom += (before_neighbors - df_clean2['prod_nom'].isna().sum())
    print(f"Total 'prod_nom' imputados en este bloque: {imputed_prod_nom}")

    # Eliminar duplicados exactos
    df_clean2 = df_clean2.drop_duplicates()

    # Recalcular missing y mostrar resumen
    missing_after2 = df_clean2.isnull().sum().rename('n_missing').to_frame()
    missing_after2['pct_missing'] = 100 * missing_after2['n_missing'] / len(df_clean2)
    display(missing_after2.sort_values('pct_missing', ascending=False))

    # Guardar intermedio variant
    inter_path = Path('intermedio')
    inter_path.mkdir(parents=True, exist_ok=True)
    out_file2 = inter_path / 'csv_intermedio_step2.csv'
    df_clean2.to_csv(out_file2, index=False)
    print('CSV intermedio (variant) guardado en:', out_file2)
else:
    print('df no definido; no se ejecutó el bloque alternativo')

Columna 'prod_marca' eliminada en df_clean2
Imputación de 'categoria' aplicada en df_clean2 (prev/next).
Total 'prod_nom' imputados en este bloque: 17075
Total 'prod_nom' imputados en este bloque: 17075


,n_missing,pct_missing
ven_id,0,0.0
fecha_venta,0,0.0
anio,0,0.0
mes,0,0.0
dia_semana,0,0.0
prod_id,0,0.0
prod_nom,0,0.0
categoria,0,0.0
precio_venta,0,0.0
ven_cantidad,0,0.0


CSV intermedio (variant) guardado en: intermedio\csv_intermedio_step2.csv


### Análisis Analítico del Proceso de Limpieza — Variante *Step 2*

Este segundo bloque de limpieza tiene como objetivo **crear una variante del dataset** (`df_clean2`) para verificar una estrategia extendida de imputación y estandarización, sin modificar el dataset previo.  
El resultado final muestra **cero valores faltantes en todas las columnas**, lo que indica que este pipeline dejó los datos completamente listos para análisis avanzado y modelado.

---

#### 1. Eliminación de columna problemática

##### **🔸 Columna eliminada:** `prod_marca`

- Se confirma que fue eliminada correctamente.
- Dado que tenía **100% de valores faltantes**, eliminación es **la decisión correcta**.
- Evita ruido innecesario en modelado y no aporta información útil.

---

#### 2. Normalización de columnas tipo string

Las columnas normalizadas fueron:

- `prod_nom`
- `categoria`
- `sucursal`
- `vendedor`
- `cliente`

Acciones aplicadas:

- `strip()` para limpiar espacios
- Conversión de valores placeholder → `NaN`  
  (`"NO POSEE"`, `"Sin Categoria"`, `"PRODUCTO NO IDENTIFICADO"`, `"nan"`)

##### **Conclusión:**
Esta estandarización era necesaria para preparar imputaciones confiables.

---

#### 3. Conversión de variables numéricas

Se convirtieron correctamente a tipo numérico las columnas:

- `precio_venta`
- `ven_cantidad`
- `ven_descuento` *(luego imputada a 0)*
- `ven_precio_unitario`
- `ingreso_neto`
- `stock_actual`
- `valor_stock_pendiente`
- `utilidad_acumulada`

**Resultado:**  
No existen valores faltantes tras la conversión → datos numéricos limpios.

---

#### 4. Imputación de *categoria*

El proceso fue **jerárquico**, utilizando:

1. `forward fill`
2. `backward fill`
3. Reglas cuando los vecinos coinciden
4. Preferencia por el valor previo si existe

**Resultado:**  
✔ Todos los valores de `categoria` fueron imputados.  
✔ Se mantiene consistencia local al imputar según contexto de filas cercanas.

---

#### 5. Imputación avanzada de *prod_nom*

Esta es la parte más importante del Step 2.

### Proceso aplicado:

##### **Fase 1 — Imputación por `prod_id` (moda):**
- Para cada `prod_id`, se buscó el nombre más frecuente (`mode()`).
- Se asignó ese nombre a todos los casos donde `prod_nom` estaba vacío.

##### **Fase 2 — Vecinos (forward + backward fill):**
- Se rellenó cualquier valor restante usando contexto secuencial.

##### **Resultado final:**
- Total 'prod_nom' imputados en este bloque: 17075
Esto corresponde exactamente al número de missing detectados antes → imputación exitosa.

##### Implicación analítica:

- El dataset ahora tiene **nombre de producto para todos los registros**, esencial para:
  - agrupar
  - modelar demanda
  - generar KPIs
  - análisis descriptivo

---

#### 6. Missing Values Finales

El reporte final muestra:

| Columna | Missing | % |
|---------|---------|------|
| Todas | 0 | 0% |

##### Estado impecable
Este es el **mejor escenario** para pasar a EDA, feature engineering o modelado.

---

#### 7. Exportación de archivo intermedio

Archivo generado: intermedio/csv_intermedio_step2.csv

Este archivo corresponde al dataset:

- sin `prod_marca`
- con imputaciones avanzadas
- completamente limpio
- sin duplicados
- sin valores faltantes

Perfecto para continuar el flujo CRISP-DM.

### Conclusiones Generales

El Step 2 representa una **versión más robusta** del proceso de limpieza:

#### Logros principales
- Eliminación correcta de una columna sin valor (`prod_marca`)
- Imputación total de `categoria`
- Imputación completa de `prod_nom` usando una estrategia mixta
- Cero valores faltantes en toda la estructura
- Dataset consistente para análisis y modelos predictivos

## Generación de nuevas columnas

In [50]:
# Generación de nuevas columnas (nombres en español) y guardado step3
inter_dir = _Path('intermedio')
# Cargar dataset resultante del paso anterior (preferir df_clean2 en memoria)
if 'df_clean2' in globals():
    df_feat = df_clean2.copy()
    print('Usando df_clean2 en memoria para generar características')
else:
    cand = inter_dir / 'csv_intermedio_step2.csv'
    if not cand.exists():
        cand = inter_dir / 'csv_intermedio_step1_variant.csv'
    if cand.exists():
        df_feat = pd.read_csv(cand, parse_dates=['fecha_venta'], low_memory=False)
        print('Leído', cand)
    else:
        raise FileNotFoundError('No se encontró csv_intermedio_step2.csv ni csv_intermedio_step1_variant.csv en intermedio/')

# Asegurar que 'fecha_venta' es datetime
df_feat['fecha_venta'] = pd.to_datetime(df_feat['fecha_venta'], errors='coerce')

# 1) Ingreso por fila: preferir 'ingreso_neto' si existe, sino calcularlo
df_feat['ingreso_calculado'] = df_feat['ingreso_neto'].fillna(df_feat.get('precio_venta',0) * df_feat.get('ven_cantidad',0))

# 2) Porcentaje de descuento implícito: (precio_venta*cantidad - ingreso_calculado) / (precio_venta*cantidad)
base = df_feat['precio_venta'].fillna(0) * df_feat['ven_cantidad'].fillna(0)
df_feat['pct_descuento'] = np.where(base>0, (base - df_feat['ingreso_calculado'].fillna(0)) / base, 0.0)
df_feat['tiene_descuento'] = df_feat['pct_descuento'] > 0

# 3) Precio unitario calculado (fallback): ingreso_calculado / cantidad o precio unitario reportado
df_feat['precio_unitario_calc'] = np.where(df_feat['ven_cantidad'].fillna(0) > 0, df_feat['ingreso_calculado'] / df_feat['ven_cantidad'].replace(0, np.nan), df_feat.get('ven_precio_unitario'))

# 4) Variables temporales en español
df_feat['es_fin_de_semana'] = df_feat['fecha_venta'].dt.dayofweek >= 5
df_feat['semana_ano'] = df_feat['fecha_venta'].dt.isocalendar().week.astype('Int64')
df_feat['mes_ano'] = df_feat['fecha_venta'].dt.to_period('M').astype(str)

# 5) Ordenar por producto y fecha para agregaciones por producto
if 'prod_id' in df_feat.columns:
    df_feat = df_feat.sort_values(['prod_id','fecha_venta'])
    df_feat['ventas_acum_prod'] = df_feat.groupby('prod_id')['ven_cantidad'].cumsum()
    df_feat['ingresos_acum_prod'] = df_feat.groupby('prod_id')['ingreso_calculado'].cumsum()
    df_feat['precio_promedio_prod'] = df_feat.groupby('prod_id')['precio_venta'].transform('mean')
    # Ventanas móviles por producto: 7 y 30 días sobre cantidad vendida
    try:
        df_feat = df_feat.set_index('fecha_venta')
        df_feat['rolling_7d_cantidad'] = df_feat.groupby('prod_id')['ven_cantidad'].rolling('7D').sum().reset_index(level=0, drop=True)
        df_feat['rolling_30d_cantidad'] = df_feat.groupby('prod_id')['ven_cantidad'].rolling('30D').sum().reset_index(level=0, drop=True)
        df_feat = df_feat.reset_index()
    except Exception as e:
        print('Advertencia: no se pudieron calcular ventanas temporales:', e)
        df_feat = df_feat.reset_index() if 'fecha_venta' in df_feat.index.names else df_feat
else:
    # Si no hay prod_id, calcular acumulados globales y dejar ventanas como NaN
    df_feat['ventas_acum_prod'] = df_feat['ven_cantidad'].cumsum()
    df_feat['ingresos_acum_prod'] = df_feat['ingreso_calculado'].cumsum()
    df_feat['precio_promedio_prod'] = df_feat['precio_venta'].mean() if 'precio_venta' in df_feat.columns else np.nan
    df_feat['rolling_7d_cantidad'] = np.nan
    df_feat['rolling_30d_cantidad'] = np.nan

# 6) Ratios relacionados con stock/valor/utilidad (nombres en español)
df_feat['ratio_valor_stock'] = np.where((df_feat['stock_actual'].notna() & df_feat['precio_venta'].notna()), df_feat['valor_stock_pendiente'] / (df_feat['stock_actual'] * df_feat['precio_venta'] + 1e-9), np.nan)
df_feat['ratio_utilidad'] = np.where(df_feat['valor_stock_pendiente'].notna(), df_feat['utilidad_acumulada'] / (df_feat['valor_stock_pendiente'] + 1e-9), np.nan)

# 7) Bandera para top10 por ingreso (en español)
if 'prod_id' in df_feat.columns and 'ingreso_calculado' in df_feat.columns:
    top10 = df_feat.groupby('prod_id')['ingreso_calculado'].sum().nlargest(10).index
    df_feat['top10_por_ingreso'] = df_feat['prod_id'].isin(top10)
else:
    df_feat['top10_por_ingreso'] = False

# 8) Guardar CSV intermedio step3 con nombres en español
inter_dir.mkdir(parents=True, exist_ok=True)
out3 = inter_dir / 'csv_intermedio_step3.csv'
df_feat.to_csv(out3, index=False)
print('CSV intermedio step3 guardado en:', out3)
# Guardar listado de nuevas columnas (en español)
cols_nuevas = [
    'ingreso_calculado','pct_descuento','tiene_descuento','precio_unitario_calc','es_fin_de_semana','semana_ano','mes_ano',
    'ventas_acum_prod','ingresos_acum_prod','precio_promedio_prod','rotacion_stock_estim','ratio_valor_stock','ratio_utilidad','top10_por_ingreso','rolling_7d_cantidad','rolling_30d_cantidad'
]

cols_exist = [c for c in cols_nuevas if c in df_feat.columns]
pd.DataFrame({'nuevas_columnas': cols_exist}).to_csv(inter_dir / 'nuevas_columnas_step3.csv', index=False)
print('Listado de nuevas columnas guardado en intermedio/nuevas_columnas_step3.csv')

Usando df_clean2 en memoria para generar características
CSV intermedio step3 guardado en: intermedio\csv_intermedio_step3.csv
Listado de nuevas columnas guardado en intermedio/nuevas_columnas_step3.csv
CSV intermedio step3 guardado en: intermedio\csv_intermedio_step3.csv
Listado de nuevas columnas guardado en intermedio/nuevas_columnas_step3.csv


### Análisis Completo del Paso 3 — Generación de Nuevas Características (Feature Engineering)

Este paso corresponde a la etapa **Feature Engineering** dentro del ciclo CRISP-DM.  
Se generan nuevas variables derivadas del comportamiento de ventas, precios, temporalidad y stock.

El objetivo es preparar el dataset para **modelos predictivos**, **EDA avanzado**, y **análisis de negocio**.

El dataset base utilizado fue: df_clean2 (desde memoria, 100% limpio)

A continuación se presenta un análisis detallado de todas las nuevas características generadas.

---

#### **A) Variables económicas**

##### **1. `ingreso_calculado`**
- Combina:
  - `ingreso_neto` (si existe)
  - `precio_venta * ven_cantidad` (como fallback)
- Asegura que toda fila tiene un valor de ingreso útil.

##### **2. `pct_descuento`**
- Cálculo:  
  \[
  \frac{(precio\_venta \times cantidad) - ingreso\_calculado}{precio\_venta \times cantidad}
  \]
- Mide el **descuento real aplicado**, no el informado.

##### **3. `tiene_descuento`**
- Booleano → `True` si `pct_descuento > 0`.

##### **4. `precio_unitario_calc`**
- Precio por unidad real pagado.
- Combina:
  - `ingreso_calculado / cantidad`
  - `ven_precio_unitario` como respaldo

---

#### **B) Variables temporales (en español)**

##### **5. `es_fin_de_semana`**
- `True` si la venta ocurrió sábado o domingo.

##### **6. `semana_ano`**
- Semana ISO del año (1–52).

##### **7. `mes_ano`**
- Periodo mensual `YYYY-MM`.

---

#### **C) Indicadores por producto**

Se aplican solo si existe `prod_id` (en este caso SÍ existe).

##### **8. `ventas_acum_prod`**
- Cantidad acumulada vendida por producto.

##### **9. `ingresos_acum_prod`**
- Ingreso acumulado por producto.

##### **10. `precio_promedio_prod`**
- Promedio histórico del precio de venta por producto.

##### **11. `rolling_7d_cantidad`**
- Suma de ventas por producto en la ventana móvil de **7 días**.

##### **12. `rolling_30d_cantidad`**
- Suma en ventana móvil de **30 días**.

Estas variables son esenciales para modelos de predicción de demanda.

---

#### 📦 **D) Indicadores relacionados al stock y utilidad**

##### **13. `ratio_valor_stock`**
- Proporción entre el valor pendiente y el valor de inventario.

##### **14. `ratio_utilidad`**
- Utilidad sobre valor pendiente.

---

#### **E) Ranking de productos por ingreso total**

##### **15. `top10_por_ingreso`**
- `True` si el producto pertenece al **Top 10 de mayor ingreso**.
- Muy útil para análisis ABC, Pareto y optimización comercial.

---
### 3. Archivos generados

#### **A) Dataset completo con nuevas columnas**

intermedio/csv_intermedio_step3.csv

#### **B) Archivo con listado de columnas nuevas**

intermedio/nuevas_columnas_step3.csv

## Detección y eliminación de Outliers

In [54]:
# Step 4: Detección y eliminación de outliers — generar csv_intermedio_step4.csv
inter_dir = _Path('intermedio')
# Cargar step3 (preferir df_feat en memoria)
if 'df_feat' in globals():
    df_s3 = df_feat.copy()
    print('Usando df_feat en memoria para detectar outliers')
else:
    p = inter_dir / 'csv_intermedio_step3.csv'
    if p.exists():
        df_s3 = pd.read_csv(p, parse_dates=['fecha_venta'], low_memory=False)
        print('Leído', p)
    else:
        raise FileNotFoundError('No se encontró csv_intermedio_step3.csv en intermedio/. Ejecuta la celda de generación step3 primero.')

# Columnas numéricas candidatas para detectar outliers (ajustar si se requiere)
candidate_numeric = ['precio_venta','ven_cantidad','ingreso_calculado','precio_unitario_calc','rotacion_stock_estim','valor_stock_pendiente','utilidad_acumulada']
num_cols = [c for c in candidate_numeric if c in df_s3.columns]
print('Columnas numéricas evaluadas para outliers:', num_cols)

# Resumen por columna usando IQR; detectamos outliers moderados (1.5*IQR) y extremos (3.0*IQR)
summary = []
extreme_mask = pd.Series(False, index=df_s3.index)
for c in num_cols:
    col = df_s3[c].dropna()
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    lower_ext = q1 - 3.0 * iqr
    upper_ext = q3 + 3.0 * iqr
    n_total = len(df_s3)
    n_missing = df_s3[c].isna().sum()
    n_moderate = ((df_s3[c] < lower) | (df_s3[c] > upper)).sum()
    n_extreme = ((df_s3[c] < lower_ext) | (df_s3[c] > upper_ext)).sum()
    summary.append({'column': c, 'n_missing': int(n_missing), 'n_moderate': int(n_moderate), 'n_extreme': int(n_extreme), 'pct_extreme': float(n_extreme / n_total * 100)})
    # Marcar filas con outliers extremos para posible eliminación
    mask_ext = (df_s3[c] < lower_ext) | (df_s3[c] > upper_ext)
    extreme_mask = extreme_mask | mask_ext.fillna(False)

# Guardar resumen de outliers por columna
outliers_df = pd.DataFrame(summary)
inter_dir.mkdir(parents=True, exist_ok=True)
outliers_df.to_csv(inter_dir / 'outliers_summary_step4.csv', index=False)
print('Resumen de outliers guardado en intermedio/outliers_summary_step4.csv')

# Cuántas filas son extremes en al menos una columna
n_extreme_rows = int(extreme_mask.sum())
total_rows = len(df_s3)
print(f'Filas con outliers extremos detectadas: {n_extreme_rows} de {total_rows} ({n_extreme_rows/total_rows*100:.4f}%)')

# Guardar muestra de filas con outliers extremos para revisión (hasta 1000 filas)
if n_extreme_rows > 0:
    sample_ext = df_s3[extreme_mask].copy()
    sample_ext.to_csv(inter_dir / 'outliers_samples_step4.csv', index=False)
    print('Muestra de outliers guardada en intermedio/outliers_samples_step4.csv')

# Política: eliminar filas que tengan outliers extremos (3*IQR) en cualquiera de las columnas evaluadas
if n_extreme_rows > 0:
    df_step4 = df_s3.loc[~extreme_mask].copy()
    print(f'Eliminando {n_extreme_rows} filas con outliers extremos. Shape antes: {total_rows}, después: {len(df_step4)}')
else:
    df_step4 = df_s3.copy()
    print('No se eliminaron filas (no se detectaron outliers extremos)')

# Guardar resultado step4
out4 = inter_dir / 'csv_intermedio_step4.csv'
df_step4.to_csv(out4, index=False)
print('CSV intermedio step4 guardado en:', out4)

# Opcional: mostrar resumen rápido
display(outliers_df)
print('Shape final step4:', df_step4.shape)

# Si quieres cambiar la política (ej: marcar en vez de eliminar), dímelo y lo actualizo

Usando df_feat en memoria para detectar outliers
Columnas numéricas evaluadas para outliers: ['precio_venta', 'ven_cantidad', 'ingreso_calculado', 'precio_unitario_calc', 'valor_stock_pendiente', 'utilidad_acumulada']
Resumen de outliers guardado en intermedio/outliers_summary_step4.csv
Filas con outliers extremos detectadas: 48160 de 180515 (26.6792%)
Muestra de outliers guardada en intermedio/outliers_samples_step4.csv
Eliminando 48160 filas con outliers extremos. Shape antes: 180515, después: 132355
Muestra de outliers guardada en intermedio/outliers_samples_step4.csv
Eliminando 48160 filas con outliers extremos. Shape antes: 180515, después: 132355
CSV intermedio step4 guardado en: intermedio\csv_intermedio_step4.csv
CSV intermedio step4 guardado en: intermedio\csv_intermedio_step4.csv


,column,n_missing,n_moderate,n_extreme,pct_extreme
0,precio_venta,0,11695,3475,1.925048
1,ven_cantidad,0,25436,15893,8.804254
2,ingreso_calculado,0,13121,5844,3.237404
3,precio_unitario_calc,0,14389,6304,3.492231
4,valor_stock_pendiente,0,36310,26880,14.890729
5,utilidad_acumulada,0,36310,26880,14.890729


Shape final step4: (132355, 37)


### Análisis de Resultados — Step 4: Detección y Eliminación de Outliers (IQR 3.0)

Este paso aplicó una política estricta de eliminación de outliers extremos utilizando el criterio **3×IQR** sobre varias columnas numéricas relevantes del dataset.

---

#### Resumen General del Proceso

- Se utilizó el dataset generado en el **step3** (`df_feat`).
- Se evaluaron **6 columnas numéricas** para detectar valores atípicos.
- Metodología aplicada: **Rango Intercuartílico (IQR)**  
  - *Outliers moderados:* fuera de 1.5×IQR  
  - *Outliers extremos:* fuera de 3.0×IQR  
- Política aplicada: **eliminar filas que presentaran al menos un outlier extremo**.

---

#### Columnas Evaluadas

precio_venta
ven_cantidad
ingreso_calculado
precio_unitario_calc
valor_stock_pendiente
utilidad_acumulada

---

#### Resultados Detectados

- **Total registros evaluados:** 180.515  
- **Filas con outliers extremos:** 48.160  
- **Proporción:** 26,68% del dataset  
- **Registros restantes después de eliminar outliers:** 132.355  

Esto indica que aproximadamente **1 de cada 4 registros contenía al menos un outlier extremo**, un porcentaje elevado pero esperable en datos transaccionales con alta variabilidad.

---

#### Detalles por Columna

| Columna                | Moderados | Extremos | % Extremos |
|------------------------|-----------|----------|------------|
| precio_venta           | 11.695    | 3.475    | 1,92%      |
| ven_cantidad           | 25.436    | 15.893   | 8,80%      |
| ingreso_calculado      | 13.121    | 5.844    | 3,23%      |
| precio_unitario_calc   | 14.389    | 6.304    | 3,49%      |
| valor_stock_pendiente  | 36.310    | 26.880   | 14,89%     |
| utilidad_acumulada     | 36.310    | 26.880   | 14,89%     |

---

#### Interpretación de los Resultados

- Las columnas con mayor volumen de outliers extremos son:  
  **valor_stock_pendiente** y **utilidad_acumulada**, ambas con cerca del 15% de registros extremos.  
- La columna **ven_cantidad** también presenta una cantidad considerable de outliers extremos (8,8%).  
- Las columnas relacionadas a precios muestran menor cantidad de extremos, lo cual es más común por su menor variabilidad.  
- El alto porcentaje de eliminación (26,68%) implica:
  -  Mejora la estabilidad estadística del dataset.  
  -  Puede reducir representatividad si parte de estos outliers reflejan situaciones reales.

---

#### Archivos Generados

- `intermedio/outliers_summary_step4.csv` – Resumen por columna  
- `intermedio/outliers_samples_step4.csv` – Muestra de outliers extremos  
- `intermedio/csv_intermedio_step4.csv` – Dataset limpio tras eliminar outliers  

---

#### Shape Final del Dataset
- Filas: 132355
- Columnas: 37

## Comprobaciones finales

In [55]:
# Comprobación: listar todas las columnas que tienen nulos o cadenas vacías en csv_intermedio_step4.csv
inter = Path('intermedio') / 'csv_intermedio_step4.csv'
if not inter.exists():
    print(f'No se encontró {inter}. Ejecuta el Step 4 antes de esta comprobación.')
else:
    df_check = pd.read_csv(inter, parse_dates=['fecha_venta'], low_memory=False)
    total_rows = len(df_check)
    results = []
    for c in df_check.columns:
        n_null = int(df_check[c].isna().sum())
        # Contar cadenas vacías o solo espacios como vacíos (aplicable también a columnas no-string)
        n_empty = int((df_check[c].astype(str).str.strip() == '').sum())
        if n_null > 0 or n_empty > 0:
            results.append({
                'columna': c,
                'n_null': n_null,
                'n_empty': n_empty,
                'pct_null': round(n_null / total_rows * 100, 3) if total_rows>0 else None,
                'pct_empty': round(n_empty / total_rows * 100, 3) if total_rows>0 else None
            })
    if not results:
        print('No se detectaron columnas con nulos o cadenas vacías en', inter)
    else:
        res_df = pd.DataFrame(results)
        display(res_df.sort_values(['n_null','n_empty'], ascending=False))

No se detectaron columnas con nulos o cadenas vacías en intermedio\csv_intermedio_step4.csv


### Comprobaciones Finales — Validación de Nulos y Cadenas Vacías

Como etapa posterior a la depuración del dataset en el **Step 4**, se ejecutó una comprobación para identificar columnas con:

- Valores nulos (`NaN`)
- Cadenas vacías (`''`) o compuestas solo por espacios

Esta verificación se aplicó al archivo:

intermedio/csv_intermedio_step4.csv

---

### Resultado de la Comprobación

El script revisó **todas las columnas** del dataset y el resultado fue:

No se detectaron columnas con nulos o cadenas vacías en intermedio/csv_intermedio_step4.csv

---

### Interpretación

- Esto confirma que **después de la detección y eliminación de outliers**, el dataset resultante quedó **completamente limpio** en términos de:
  - Falta de valores
  - Celdas vacías
  - Celdas con espacios en blanco
- Todas las columnas contienen información válida en el 100% de sus filas.
- Este es un excelente indicador de **calidad de datos** previo a procesos de modelado, análisis estadístico o etapas posteriores del pipeline CRISP-DM.


## Conclusiones de la fase 03 Data Preparation

Esta fase consolidó un proceso completo de limpieza, depuración, enriquecimiento y validación del dataset, asegurando que los datos estén en óptimas condiciones para el análisis avanzado y el modelado predictivo. A continuación, se presenta un resumen claro y ordenado de todo el flujo aplicado, los hallazgos y los próximos pasos recomendados.

---

### Resumen del Flujo Aplicado

**1. Carga y exploración inicial**  
- Lectura del archivo bruto: `bruto/csv_ferredash_raw.csv`.  
- Revisión de estructura, tipos, nulos y valores inconsistentes para definir el plan de limpieza.

**2. Limpieza segura — Step 1**  
- Normalización de cadenas (espacios, mayúsculas/minúsculas).  
- Reemplazo de placeholders como `"NO POSEE"`, `"Sin Categoria"` y `"PRODUCTO NO IDENTIFICADO"` → `NaN`.  
- Conversión segura a valores numéricos con `to_numeric(errors='coerce')`.  
- Imputación simple (ej. descuentos = 0).  
- Eliminación de duplicados.  
- Guardado en: `intermedio/csv_intermedio_step1.csv`.

**3. Limpieza profunda — Step 2**  
- Eliminación de columna `prod_marca` (100% faltantes).  
- Imputación jerárquica de `categoria` (forward/backward + reglas específicas).  
- Imputación avanzada de `prod_nom` usando:
  - Moda por `prod_id`.
  - Relleno basado en vecinos.  
- Guardado en: `intermedio/csv_intermedio_step2.csv`.

**4. Feature Engineering — Step 3**  
- Creación de variables económicas: `ingreso_calculado`, `pct_descuento`, `precio_unitario_calc`.  
- Variables temporales: `es_fin_de_semana`, `semana_ano`, `mes_ano`.  
- Agregaciones por producto: acumulados, ventanas móviles, ratios de utilidad y stock.  
- Guardado en: `intermedio/csv_intermedio_step3.csv`.  
- Listado de nuevas columnas en: `intermedio/nuevas_columnas_step3.csv`.

**5. Detección y eliminación de outliers — Step 4**  
- Evaluación por IQR (1.5× y 3×).  
- Eliminación de filas con outliers extremos (≥3×IQR).  
- Archivos generados:
  - `intermedio/csv_intermedio_step4.csv`
  - `outliers_summary_step4.csv`
  - `outliers_samples_step4.csv`

---

### Hallazgos Principales

- La columna **`prod_marca`** estaba completamente vacía → decisión correcta eliminarla.  
- **`categoria`** y **`prod_nom`** tenían faltantes significativos → las imputaciones aplicadas permitieron recuperar gran parte de la información.  
- El Step 3 generó un conjunto robusto de **variables derivadas** altamente útiles para modelado.  
- En Step 4 se eliminaron filas con outliers extremos, lo que mejora estabilidad estadística, aunque podría afectar representatividad (se recomienda evaluar impacto).

---

### Acciones Realizadas sobre Archivos

Se generaron y almacenaron archivos de respaldo en `intermedio/`:

- `csv_intermedio_step1.csv`
- `csv_intermedio_step2.csv`
- `csv_intermedio_step3.csv`
- `csv_intermedio_step4.csv`
- `outliers_summary_step4.csv`
- `outliers_samples_step4.csv`
- `nuevas_columnas_step3.csv`

Otros procesos añadidos:

- **Comprobación automatizada** de columnas con nulos o cadenas vacías sobre Step 4.  
- **Eliminación de columnas auxiliares** utilizadas solo para ingeniería temporal (`rotacion_stock_estim`, `fecha_venta_prev`, `dias_desde_venta_prev`).  
- **Copia del CSV procesado** hacia `procesado/csv_procesado.csv`, manteniendo intacto el Step 4 en `intermedio/`.

---

### Recomendaciones y Próximos Pasos

- **Revisar política de outliers:** analizar ejemplos en `outliers_samples_step4.csv` para decidir si conviene mantener la política de eliminación o aplicar winsorización/marcado.  
- **Auditar imputaciones críticas:** revisar casos imputados para `prod_nom` y `categoria` para evitar sesgos o sobreimputación.  
- **Documentar las transformaciones** (regex, reglas, columnas eliminadas) en un README para trazabilidad.  
- **Generar metadatos finales:** crear un JSON con:
  - Shape final  
  - Filas eliminadas  
  - Columnas imputadas/eliminadas  
  - Parámetros utilizados  
- **Construir un pipeline reproducible:** encapsular Step1 → Step4 en scripts o módulos ejecutables.  
- **Seleccionar features clave para modelado:** validar relevancia de:
  - `ingreso_calculado`
  - `ventas_acum_prod`
  - `rolling_7d_cantidad`
  - `precio_promedio_prod`
  - `top10_por_ingreso`, etc.

---

### Indicadores de Calidad Final

- El dataset final presenta **shape consistente**, acorde a la eliminación de outliers.  
- La comprobación de nulos/vacíos muestra **ausencia total de faltantes**, un indicador clave de calidad para modelado.  

---

### Notas Finales

Las decisiones aplicadas fueron técnicamente sólidas y coherentes con los objetivos analíticos:

- Eliminación de información irrelevante (`prod_marca`).  
- Identificación y eliminación de outliers extremos.  
- Remoción de columnas auxiliares.  
- Generación de un dataset procesado y limpio en `procesado/csv_procesado.csv`.

El proceso deja un dataset robusto, consistente y bien preparado para avanzar a la fase de **Modelado (Fase 4)**.

#### Crear Archivo procesado final para la siguiente fase de modelado

In [ ]:
# Copiar y renombrar el CSV procesado a la carpeta `procesado/` sin eliminar el original
inter = Path('intermedio') / 'csv_intermedio_step4.csv'
proc_dir = Path('procesado')

if not inter.exists():
    print(f'No se encontró {inter}. Ejecuta Step 4 antes de copiar a procesado.')
else:
    # Crear carpeta procesado/ si no existe
    proc_dir.mkdir(parents=True, exist_ok=True)
    
    # Nuevo nombre del archivo en procesado/
    dest = proc_dir / 'csv_procesado.csv'
    
    try:
        shutil.copy2(str(inter), str(dest))
        print('Archivo copiado y renombrado en procesado como:', dest)
    except Exception as e:
        print('Error al copiar el archivo:', e)

Archivo copiado y renombrado en procesado como: procesado\csv_procesado.csv
